# Magnetic Resonance Imaging - 361.2.6501
## Final Project: Central-Slice Brain Tumor Segmentation

**Students:** Yuval Ratzabi (ID: TODO), Second student (ID: TODO)

This notebook evaluates the five GMM pipelines with leakage-safe five-fold nested patient-level evaluation. Every patient is held out exactly once as outer-test data.

## 1. Evaluation protocol

For each outer fold, the other four folds are divided into training and validation patients. GMMs are fitted only on the training patients, all probability and image-processing parameters are selected only on validation patients, and the outer-test patients are evaluated once. The reported result is the mean and standard deviation across the five held-out folds.

The Combined hierarchy is optional. It is retained only when it improves validation Dice over Boundary + Symmetry by at least 0.005; otherwise Combined uses the exact no-hierarchy fusion fallback. No outer-test result participates in this choice.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from config import CV_OUTPUT_DIR, FIGURES_DIR, MODEL_NAMES
from data.preprocessing import load_preprocessed_slice
from data.splits import create_nested_cv_folds
from evaluation.cross_validation import evaluate_saved_fold_models, run_nested_cross_validation
from evaluation.visualizations import choose_qualitative_examples, plot_pipeline_diagnostic, plot_qualitative_examples, plot_required_scatterplots

%matplotlib inline
sns.set_theme(style="whitegrid")
Path(CV_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

N_BASELINE_TRIALS = 30
N_ADVANCED_TRIALS = 20
FORCE_NEW_FOLDS = False
FORCE_RETRAIN_FOLD_MODELS = False
REUSE_COMPLETED_FOLDS = True

## 2. Five outer folds

The manifests are saved once and reused. `FORCE_NEW_FOLDS=True` intentionally creates a new evaluation protocol and should not be used after results have been inspected.

In [ ]:
folds = create_nested_cv_folds(force=FORCE_NEW_FOLDS)
split_rows = []
for fold in folds:
    for role in ("train", "validation", "test"):
        sizes = np.asarray([load_preprocessed_slice(int(volume_id)).whole_tumor.sum() for volume_id in fold[role]])
        split_rows.append({
            "fold": fold["fold"], "role": role, "patients": len(sizes),
            "tumor_present": int((sizes > 0).sum()), "tumor_free": int((sizes == 0).sum()),
            "median_tumor_pixels": float(np.median(sizes[sizes > 0])) if np.any(sizes > 0) else 0.0,
        })
split_summary = pd.DataFrame(split_rows)
outer_test_ids = np.concatenate([fold["test"] for fold in folds])
assert len(outer_test_ids) == len(np.unique(outer_test_ids)) == 369
display(split_summary)

## 3. Nested cross-validation

This is the long-running cell. Each completed fold is saved independently, so an interrupted run can resume with `REUSE_COMPLETED_FOLDS=True`. Fold-specific GMM directories prevent models trained on one split from being reused in another split.

In [ ]:
cv_results = run_nested_cross_validation(
    folds,
    n_baseline_trials=N_BASELINE_TRIALS,
    n_advanced_trials=N_ADVANCED_TRIALS,
    force_retrain_models=FORCE_RETRAIN_FOLD_MODELS,
    reuse_completed_folds=REUSE_COMPLETED_FOLDS,
)

## 4. Validation versus held-out performance

The table compares the average inner-validation score with the average outer-test score. The outer-test mean and fold-to-fold standard deviation are the primary generalization result.

In [ ]:
generalization = cv_results["generalization_summary"].copy()
display(generalization.round(4))
fold_scores = cv_results["outer_test_fold_summaries"]
display(fold_scores.pivot(index="fold", columns="model", values="dice_mean").round(4))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=cv_results["validation_fold_summaries"], x="model", y="dice_mean", ax=axes[0])
sns.stripplot(data=cv_results["validation_fold_summaries"], x="model", y="dice_mean", color="black", size=4, ax=axes[0])
sns.boxplot(data=fold_scores, x="model", y="dice_mean", ax=axes[1])
sns.stripplot(data=fold_scores, x="model", y="dice_mean", color="black", size=4, ax=axes[1])
axes[0].set_title("Inner-validation Dice across folds")
axes[1].set_title("Held-out outer-test Dice across folds")
for axis in axes:
    axis.set_xlabel("")
    axis.tick_params(axis="x", rotation=20)
figure.tight_layout()
figure.savefig(Path(FIGURES_DIR) / "nested_cv_validation_vs_test.png", dpi=300, bbox_inches="tight")

## 5. Out-of-fold failures

Every row below is a prediction made while that patient was held out from GMM fitting and parameter selection.

In [ ]:
outer_predictions = cv_results["outer_test_per_volume"]
failure_rows = []
for model_name in MODEL_NAMES:
    frame = outer_predictions[outer_predictions["model"] == model_name]
    failure_rows.append({
        "model": model_name,
        "missed_tumors": frame.loc[frame["missed_tumor"], "volume_id"].astype(int).tolist(),
        "empty_slice_false_positives": frame.loc[frame["empty_slice_false_positive"], "volume_id"].astype(int).tolist(),
    })
display(pd.DataFrame(failure_rows))

## 6. Baseline versus proposed model

Combined denotes hierarchy when selected by the corresponding fold's validation data and the exact Boundary + Symmetry fallback otherwise.

In [ ]:
baseline_frame = outer_predictions[outer_predictions["model"] == "Raw (4D)"].copy()
proposed_frame = outer_predictions[outer_predictions["model"] == "Combined"].copy()
plot_required_scatterplots(baseline_frame, proposed_frame, Path(FIGURES_DIR) / "nested_cv_baseline_vs_combined.png")

## 7. Qualitative examples and pipeline inspection

Each selected volume is reconstructed with the model and parameters from its own held-out outer fold.

In [ ]:
selected_examples = choose_qualitative_examples(baseline_frame, proposed_frame)
display(pd.DataFrame({"category": selected_examples.keys(), "volume_id": selected_examples.values()}))
baseline_details, proposed_details = {}, {}
for volume_id in [value for value in selected_examples.values() if value is not None]:
    fold_index = int(proposed_frame.loc[proposed_frame["volume_id"] == volume_id, "fold"].iloc[0])
    fold = folds[fold_index - 1]
    details = evaluate_saved_fold_models(fold, [volume_id], ("Raw (4D)", "Combined"))
    baseline_details[volume_id] = details["Raw (4D)"]["details"][volume_id]
    proposed_details[volume_id] = details["Combined"]["details"][volume_id]
plot_qualitative_examples(selected_examples, baseline_details, proposed_details, Path(FIGURES_DIR) / "nested_cv_qualitative_examples.png")

In [ ]:
worst_tumor = proposed_frame[proposed_frame["tumor_present"]].sort_values("dice").iloc[0]
worst_volume = int(worst_tumor["volume_id"])
worst_fold = folds[int(worst_tumor["fold"]) - 1]
worst_details = evaluate_saved_fold_models(worst_fold, [worst_volume], ("Combined",))["Combined"]["details"][worst_volume]
print(f"Worst held-out Combined case: volume {worst_volume}, fold {int(worst_tumor['fold'])}, Dice {worst_tumor['dice']:.4f}")
display(pd.DataFrame(worst_details["component_table"]))
plot_pipeline_diagnostic(worst_details, Path(FIGURES_DIR) / f"nested_cv_worst_volume_{worst_volume}.png")

## 8. Conclusions

After the run, summarize the outer-test mean ± standard deviation, the remaining validation-to-test gap, the number of hierarchy-selected folds, missed tumors, empty-slice false positives, and whether Combined improves on Raw and Boundary + Symmetry.